In [1]:
# ======================
# 1. Imports
# ======================
import pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

2025-09-13 17:38:14.211441: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757785094.406000      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757785094.463867      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# # ======================
# # 1. Imports
# # ======================
# import pandas as pd
# from PIL import Image
# from transformers import AutoProcessor, AutoModelForCausalLM
# import torch

In [3]:
# ======================
# 2. Load CSV
# ======================
csv_path = "/kaggle/input/hp-all/hp_all_cap.csv"   # your CSV
df = pd.read_csv(csv_path)

# Explicitly use the "image_path" column
print("Columns in CSV:", df.columns.tolist())
print(df.head())

Columns in CSV: ['image_path', 'caption']
                                          image_path  \
0  /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP...   
1  /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP...   
2  /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP...   
3  /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP...   
4  /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP...   

                                             caption  
0              Owl sitting on pivet drive sign board  
1            Professor Dumbledore under street light  
2  Professor Dumbledore and Professor Mcgonagall ...  
3                    Baby with lightning shaped scar  
4                      Harry peeking though trapdoor  


In [4]:
# ======================
# 3. Load Pretrained Model
# ======================
model_id = "Salesforce/blip-image-captioning-large"   # change to BLIP-large or GIT-base later
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(model_id).to("cuda")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

In [5]:
# # ======================
# # 3. Load Pretrained Model (GIT-base)
# # ======================
# model_id = "nlpconnect/vit-gpt2-image-captioning"
# processor = AutoProcessor.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(model_id).to("cuda")

In [6]:
# ======================
# 4. Run Inference
# ======================
captions = []
for i, row in df.iterrows():
    try:
        image_path = row["image_path"]  # <- use column name directly
        image = Image.open(image_path).convert("RGB")

        inputs = processor(image, return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_length=30)
        caption = processor.decode(out[0], skip_special_tokens=True)

        captions.append(caption)
    except Exception as e:
        print(f"Error on {row['image_path']}: {e}")
        captions.append("ERROR")

Error on /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP4/gobletoffire104.jpg: [Errno 2] No such file or directory: '/kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP4/gobletoffire104.jpg'
Error on /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP7/deathlyhallowsone154.jpg: [Errno 2] No such file or directory: '/kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP7/deathlyhallowsone154.jpg'


In [7]:
# # ======================
# # 4. Run Inference
# # ======================
# captions = []
# for i, row in df.iterrows():
#     try:
#         image_path = row["image_path"]  # <- use column name directly
#         image = Image.open(image_path).convert("RGB")

#         inputs = processor(images=image, return_tensors="pt").to("cuda")
#         out = model.generate(**inputs, max_length=30)
#         caption = processor.batch_decode(out, skip_special_tokens=True)[0]

#         captions.append(caption)
#     except Exception as e:
#         print(f"Error on {row['image_path']}: {e}")
#         captions.append("ERROR")

In [8]:
# ======================
# 5. Save Captions
# ======================
out_df = df[["image_path"]].copy()   # keep only image_path
out_df["generated_caption"] = captions  # add generated captions

out_path = "/kaggle/working/generated_captions_bl.csv"
out_df.to_csv(out_path, index=False)

print("✅ Captions saved to", out_path)

✅ Captions saved to /kaggle/working/generated_captions_bl.csv
